# Issue 002 — YOLO26 + ByteTrack perception on URFD

End-to-end notebook for the shared perception front-end:

1. Mount Drive and confirm the persistent project layout exists.
2. Stage URFD from Kaggle into `MyDrive/fall_detection/datasets/urfd/`.
   Kaggle credentials are read from **Colab Secrets** (`KAGGLE_USERNAME`,
   `KAGGLE_KEY`) and never written to disk or printed.
3. Build the URFD debug manifest from the staged folder tree.
4. Run YOLO26 + ByteTrack on every debug clip's ordered frame folder.
5. Write per-clip artefacts (detections, tracks, report, run meta) to
   `artifacts/perception/<clip_id>/` on Drive.
6. Render annotated frames so a human can review track-through-fall.

Re-running is safe: step 2 short-circuits when URFD is already on Drive;
step 4 overwrites the artefact folder for each clip cleanly.

Issue 002 hard rules (enforced by `perception/tracker.py`):
  - Detector must be `yolo26m`; anything else aborts loudly.
  - Tracker must be `bytetrack.yaml` with `persist=True`.
  - Pretrained weights only — no training.
  - URFD has no ground truth, so formal mAP / IDF1 / MOTA / HOTA are
    recorded as `n/a (no ground truth)`. We validate via annotated
    review and the track-continuity report.

## 1. Mount Drive + project layout

Reuses `colab/setup.py` from Issue 001 — no new layout work here.

In [ ]:
import os, sys, pathlib

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not on Colab — Drive mounting skipped.")

PROJECT_ROOT = pathlib.Path(os.environ.get("FALL_DETECTION_PROJECT_ROOT", "/content/fall_detection"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from colab.setup import DriveLayout
layout = DriveLayout.resolve()
layout.ensure()
print("Drive layout:")
for name in ("datasets", "artifacts", "logs"):
    print(f"  {name:<10} -> {getattr(layout, name)}")

## 2. Install kagglehub

The Issue 002 dependency (kagglehub) is added to `APPROVED_PIP_PACKAGES`
in Issue 001. We re-run the install here so this notebook works even
if the user opens it before Issue 001's setup notebook.

torch / torchvision are intentionally NOT touched.

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "kagglehub"],
    check=True,
)
print("kagglehub installed.")

## 3. Set up Kaggle credentials from Colab Secrets

**Before running this cell**, add two secrets via the Colab Secrets UI
(the key-shaped icon on the left sidebar):

  - `KAGGLE_USERNAME` — your Kaggle username
  - `KAGGLE_KEY`      — your Kaggle API key

The values stay in Colab Secrets. The cell below sets two environment
variables from those secrets and never logs or persists the values.

If you are not on Colab, the staging script will look for a local
`.kaggle/kaggle.json` instead.

In [ ]:
if IN_COLAB:
    from google.colab import userdata
    username = userdata.get("KAGGLE_USERNAME")
    key = userdata.get("KAGGLE_KEY")
    if not username or not key:
        raise RuntimeError(
            "Kaggle credentials missing in Colab Secrets.\n"
            "Add KAGGLE_USERNAME and KAGGLE_KEY in the Secrets panel, "
            "then re-run this cell."
        )
    os.environ["KAGGLE_USERNAME"] = username
    os.environ["KAGGLE_KEY"] = key
    print("Kaggle credentials loaded from Colab Secrets.")
else:
    print("Not on Colab — assuming ~/.kaggle/kaggle.json is configured.")

## 4. Stage URFD from Kaggle into Drive

Idempotent: short-circuits when `datasets/urfd/.staged_from_kaggle.txt`
already exists. The marker file proves the tree came from the staging
script (not a stray hand-placed copy).

In [ ]:
from data.stage_urfd import stage_urfd_from_kaggle

staging = stage_urfd_from_kaggle(layout.root)
print(f"URFD staged at: {staging.staged_root}")
print(f"  slug           : {staging.kaggle_slug}")
print(f"  clip folders   : {staging.clip_count}")
print(f"  already staged : {staging.already_staged}")
print()
print("Sample clip folders:")
for clip in staging.clip_folders[:6]:
    print(f"  {clip.folder_name:<24}  label={clip.label:<8}  camera={clip.camera}  seq={clip.clip_sequence}")

## 5. Build the URFD debug manifest

The manifest builder walks the staged tree, parses each folder name
(`fall-NN-camM` → fall, `adl-NN-camM` → no_fall), and emits a
schema-1.1 manifest. The manifest is validated before write so a
broken staged tree can never produce a broken manifest.

In [ ]:
from data.build_urfd_manifest import build_urfd_manifest, write_urfd_manifest

manifest = build_urfd_manifest(staging.staged_root)
manifest_path = staging.staged_root / "manifest.yaml"
write_urfd_manifest(manifest, manifest_path)
print(f"Manifest written: {manifest_path}")
print(f"  schema version : {manifest.schema_version}")
print(f"  clip rows      : {len(manifest.clips)}")
fall_count = sum(1 for c in manifest.clips if c.label.value == "fall")
no_fall_count = sum(1 for c in manifest.clips if c.label.value == "no_fall")
print(f"  fall labels    : {fall_count}")
print(f"  no_fall labels : {no_fall_count}")

## 6. Run YOLO26 + ByteTrack on every debug clip

For each clip:

  1. Resolve the frame folder on Drive.
  2. Order frames numerically via `FrameFolderReader` (CRITICAL —
     out-of-order frames produce invalid tracks).
  3. Run `run_tracker_on_folder(clip_id, ordered_paths, config)`.
  4. Build a `TrackContinuityReport` and write the per-clip artefacts.
  5. Render annotated frames so a human can review track-through-fall.

Run starts at `max_frames_per_clip` per clip for the debug tier so a
session doesn't blow its wall-clock budget. Raise the limit for the
second pass once you trust the wiring.

In [ ]:
from perception.frames import FrameFolderReader
from perception.tracker import (
    TrackerConfig,
    run_tracker_on_folder,
    query_gpu_name,
)
from perception.report import build_track_continuity_report
from perception.artifacts import (
    write_perception_artifacts,
    detections_grouped_by_frame,
)
from perception.render import render_annotated_clip
from PIL import Image
import numpy as np

MAX_FRAMES_PER_CLIP = 120  # debug-tier cap; raise after wiring is trusted
ARTIFACTS_ROOT = layout.artifacts / "perception"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"GPU detected: {query_gpu_name() or 'none (CPU-only runtime)'}")
print()

config = TrackerConfig()
summary_rows = []

for clip in manifest.clips:
    clip_folder = layout.root / clip.source_path
    if not clip_folder.is_dir():
        print(f"[skip] {clip.clip_id}: folder missing on Drive: {clip_folder}")
        continue

    reader = FrameFolderReader(clip_folder)
    ordered = reader.frames()
    if not ordered:
        print(f"[skip] {clip.clip_id}: no frames in {clip_folder}")
        continue

    # Debug-tier cap: take the first N frames so sessions stay bounded.
    capped = ordered[:MAX_FRAMES_PER_CLIP]
    paths = [frame.path for frame in capped]

    print(f"[run]  {clip.clip_id}: {len(paths)} frames from {clip_folder.name}")
    run = run_tracker_on_folder(clip.clip_id, paths, config=config)
    run.source_folder = str(clip_folder.relative_to(layout.root))

    report = build_track_continuity_report(run, source_folder=run.source_folder)
    out_dir = ARTIFACTS_ROOT / clip.clip_id
    paths_written = write_perception_artifacts(out_dir, run, report)

    # Render annotated frames for visual review.
    images = [np.array(Image.open(p).convert("RGB")) for p in paths]
    detections_by_frame = detections_grouped_by_frame(run.detections, run.frame_count)
    annotated_paths = render_annotated_clip(
        images, detections_by_frame,
        output_dir=out_dir / "annotated",
        clip_id=clip.clip_id,
    )

    summary_rows.append({
        "clip_id": clip.clip_id,
        "frames": run.frame_count,
        "detections": run.detection_count,
        "tracks": run.track_count,
        "longest_track_id": report.longest_track_id,
        "longest_track_length": report.longest_track_length,
        "id_switch_count": report.id_switch_count,
        "fps": round(report.fps, 2),
        "latency_ms_per_frame": round(report.latency_ms_per_frame, 2),
    })
    print(f"       → {report.detection_count} detections, "
          f"{report.track_count} tracks, longest=track {report.longest_track_id} "
          f"({report.longest_track_length} frames), "
          f"{report.id_switch_count} id switches, {report.fps:.1f} fps")
    print(f"       → artefacts: {out_dir}")

## 7. Per-clip summary

The summary rows show what each clip's run produced. Two human-review
questions to ask:

  1. **Did the falling person keep the same track_id through the fall
     window?** Look at `longest_track_id` for the fall clips and
     cross-check the annotated frames in `artifacts/perception/<clip_id>/annotated/`.

  2. **Did ID switches happen at the moment of collapse?** Look at
     `id_switch_count` for fall clips — non-zero on a fall clip is
     a known issue per the PRD's *Further Notes* (cheapest killer
     risk: tracking through the fall).

If a fall clip shows zero detections, or if the longest track does
NOT span the fall, **apply the fallback levers** in this order:

  1. Lower `track_low_thresh` (more permissive association).
  2. Switch to BoT-SORT (`fallback_tracker='botsort.yaml'`).
  3. Try `fallback_end2end=False` (BoT-SORT only).

Each lever is exposed on `TrackerConfig`; re-run just this cell with
the chosen fallback set. Record which lever was used in the run meta
JSON the tracker writes.

In [ ]:
import json
from pathlib import Path

summary_path = ARTIFACTS_ROOT / "_run_summary.json"
summary_path.write_text(json.dumps({
    "gpu_name": query_gpu_name(),
    "max_frames_per_clip": MAX_FRAMES_PER_CLIP,
    "config": {
        "model_name": config.model_name,
        "tracker_config": config.tracker_config,
        "confidence_threshold": config.confidence_threshold,
    },
    "metric_availability": {
        "map_50": "n/a (no detection ground truth)",
        "idf1": "n/a (no tracking ground truth)",
        "mota": "n/a (no tracking ground truth)",
        "hota": "n/a (no tracking ground truth)",
    },
    "clips": summary_rows,
}, indent=2), encoding="utf-8")
print(f"Run summary written: {summary_path}")
print()
print("Per-clip summary:")
print(f"  {'clip_id':<32} {'frames':>7} {'dets':>6} {'tracks':>7} {'longest':>8} {'switches':>9} {'fps':>6}")
for row in summary_rows:
    print(f"  {row['clip_id']:<32} {row['frames']:>7} {row['detections']:>6} "
          f"{row['tracks']:>7} {str(row['longest_track_id']):>8} {row['id_switch_count']:>9} {row['fps']:>6.1f}")

## 8. Done

Issue 002 deliverables on Drive:

  - `MyDrive/fall_detection/datasets/urfd/` — staged URFD frames.
  - `MyDrive/fall_detection/datasets/urfd/manifest.yaml` — URFD debug
    manifest (schema 1.1).
  - `MyDrive/fall_detection/artifacts/perception/<clip_id>/` — per clip:
    - `<clip_id>_detections.json` — flat detection list.
    - `<clip_id>_tracks.json` — per-track summary.
    - `<clip_id>_report.json` — track-continuity report.
    - `<clip_id>_run_meta.json` — model / tracker / timing metadata.
    - `annotated/<clip_id>_frame_NNNNN.png` — boxes + track IDs drawn.
  - `MyDrive/fall_detection/artifacts/perception/_run_summary.json`
    — human-readable per-clip summary across the whole run.

Next step readiness:
  - If the falling person keeps the same track_id through the fall
    window on the debug tier → Issue 003 (clip generator) can proceed.
  - If ID switches happen at the moment of collapse on most fall clips →
    escalate by trying fallback levers before moving on.